In [1]:
!pip install -U beautifulsoup4 pypdf langgraph langserve[all] langchain_community chromadb langchain langchain_openai openai tiktoken rank_bm25 sentence_transformers cohere langchain_cohere flashrank faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.7/144.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.4/644.4 kB 31.5 MB/s eta 0:

In [2]:
import os
import warnings
warnings.filterwarnings("ignore")
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda
from langchain.schema.output_parser import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_community.tools import TavilySearchResults
from langchain_core.messages import HumanMessage, SystemMessage, trim_messages
from langgraph.prebuilt import create_react_agent, ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph, MessagesState
from langchain_core.tools import tool
from typing import Literal
from google.colab import userdata
import bs4
from langchain.vectorstores import Chroma
import chromadb
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.messages import HumanMessage, AIMessage

In [3]:
OPENAI_API_TOKEN=userdata.get('OPENAI_API_KEY')
LANGCHAIN_API_KEY = userdata.get('LANGCHAIN_API_KEY')
LANGSMITH_TRACING = userdata.get('LANGSMITH_TRACING')
LANGSMITH_ENDPOINT = userdata.get('LANGSMITH_ENDPOINT')
LANGSMITH_PROJECT = userdata.get('LANGSMITH_PROJECT')
os.environ["OPENAI_API_KEY"] = OPENAI_API_TOKEN
os.environ["LANGCHAIN_API_KEY"] = LANGCHAIN_API_KEY
os.environ["LANGSMITH_TRACING"] = LANGSMITH_TRACING
os.environ["LANGSMITH_ENDPOINT"] = LANGSMITH_ENDPOINT
os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI()
parser = StrOutputParser()

ConversationBufferMemory

In [58]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferWindowMemory
from langchain.memory import ConversationEntityMemory
from langchain.memory import ConversationSummaryMemory
from langchain.memory import ChatMessageHistory
from langchain.memory import ConversationSummaryBufferMemory
from langchain.memory.prompt import ENTITY_MEMORY_CONVERSATION_TEMPLATE
from pydantic import BaseModel
from typing import List, Dict, Any

In [5]:
memory = ConversationBufferMemory()

<ipython-input-5-a33ea3a4cc0d>:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()


In [6]:
memory.save_context({"input": "Hi"},
                    {"output": "What's up"})

In [7]:
memory.load_memory_variables({})

{'history': "Human: Hi\nAI: What's up"}

In [8]:
memory2 = ConversationBufferMemory(return_messages=True)

memory2.save_context({"input": "Hi"},
                    {"output": "What's up"})

memory2.load_memory_variables({})

{'history': [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}),
  AIMessage(content="What's up", additional_kwargs={}, response_metadata={})]}

In [9]:
conversation = ConversationChain(llm=llm,verbose=True,memory=ConversationBufferMemory())

<ipython-input-9-ae553430dac7>:1: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use :meth:`~RunnableWithMessageHistory: https://python.langchain.com/v0.2/api_reference/core/runnables/langchain_core.runnables.history.RunnableWithMessageHistory.html` instead.
  conversation = ConversationChain(llm=llm,verbose=True,memory=ConversationBufferMemory())


In [10]:
conversation.predict(input="Hi there!")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi there!
AI:

> Finished chain.


'Hello! How are you today?'

In [11]:
conversation.predict(input="Nothing much! Just tell me how do a conversation with an AI.")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi there!
AI: Hello! How are you today?
Human: Nothing much! Just tell me how do a conversation with an AI.
AI:

> Finished chain.


"Oh, that's great to hear! To have a conversation with an AI like me, you simply just start talking and I'll do my best to respond. You can ask me questions, share your thoughts and feelings, or even just chat about random topics. I'll use my programming and data to provide you with helpful and informative responses. So feel free to ask me anything you'd like to know!"

In [12]:
conversation.memory.chat_memory.messages

[HumanMessage(content='Hi there!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hello! How are you today?', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Nothing much! Just tell me how do a conversation with an AI.', additional_kwargs={}, response_metadata={}),
 AIMessage(content="Oh, that's great to hear! To have a conversation with an AI like me, you simply just start talking and I'll do my best to respond. You can ask me questions, share your thoughts and feelings, or even just chat about random topics. I'll use my programming and data to provide you with helpful and informative responses. So feel free to ask me anything you'd like to know!", additional_kwargs={}, response_metadata={})]

ConversationBufferWindowMemory


In [13]:
window_memory = ConversationBufferWindowMemory(k=1)

<ipython-input-13-48eb35e40d0d>:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  window_memory = ConversationBufferWindowMemory(k=1)


In [14]:
window_memory.save_context(
    {"input": "Hi"},
    {"output": "What's up"}
)
window_memory.save_context(
    {"input": "Not much, just hanging"},
    {"output": "Cool"}
)

In [15]:
window_memory.load_memory_variables({})

{'history': 'Human: Not much, just hanging\nAI: Cool'}

In [16]:
window_memory = ConversationBufferWindowMemory( k=2, return_messages=True)
window_memory.save_context(
    {"input": "Hi"},
    {"output": "What's up"}
)
window_memory.save_context(
    {"input": "Not much, just hanging"},
    {"output": "Cool"}
)
window_memory.save_context(
    {"input": "ok thanks "},
    {"output": "great thankyou"}
)

In [17]:
window_memory.load_memory_variables({})

{'history': [HumanMessage(content='Not much, just hanging', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Cool', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='ok thanks ', additional_kwargs={}, response_metadata={}),
  AIMessage(content='great thankyou', additional_kwargs={}, response_metadata={})]}

In [18]:
conversation_window = ConversationChain(
    llm=llm,
    memory=window_memory,
    verbose=True
)

In [19]:
conversation_window.predict(input="Hey! What's up?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
[HumanMessage(content='Not much, just hanging', additional_kwargs={}, response_metadata={}), AIMessage(content='Cool', additional_kwargs={}, response_metadata={}), HumanMessage(content='ok thanks ', additional_kwargs={}, response_metadata={}), AIMessage(content='great thankyou', additional_kwargs={}, response_metadata={})]
Human: Hey! What's up?
AI:

> Finished chain.


'Not much, just hanging.'

Conversation Entity Memory

In [22]:
memory = ConversationEntityMemory(llm=llm)

/usr/local/lib/python3.11/dist-packages/pydantic/main.py:253: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


In [23]:
_input= {"input": "i am very hungry."}
memory.load_memory_variables(_input)

{'history': '', 'entities': {}}

In [24]:
_input= {"input": "sunny & mayank are working on a hackathon project"}
memory.load_memory_variables(_input)

{'history': '', 'entities': {'sunny': '', 'Mayank': ''}}

In [25]:
memory.save_context(
    {"Human": "Sunny and Mayank are working on a hackathon project"},
    {"AI": "That's awesome! What's the hackathon project about?"}
)

In [29]:
memory.load_memory_variables({"input": "what is sunny working on?"})

{'history': "Human: Sunny and Mayank are working on a hackathon project\nAI: That's awesome! What's the hackathon project about?",
 'entities': {'Sunny': ''}}

In [30]:
memory.save_context(
    {"Human": "It's a machine learning project focused on healthcare."},
     {"AI": "Sounds exciting! Are they building a prediction model or something else?"}
)
memory.save_context(
    {"Human": "Yes, they are building prediction model."},
    {"AI": "Wishing Sunny and Mayank all the best for their project!"}
)

In [31]:
memory.load_memory_variables({"input": "who is Sunny?"})

{'history': "Human: Sunny and Mayank are working on a hackathon project\nAI: That's awesome! What's the hackathon project about?\nHuman: It's a machine learning project focused on healthcare.\nAI: Sounds exciting! Are they building a prediction model or something else?\nHuman: Yes, they are building prediction model.\nAI: Wishing Sunny and Mayank all the best for their project!",
 'entities': {'Sunny': 'Sunny is working on a machine learning project focused on healthcare and is building a prediction model.'}}

In [32]:
memory.load_memory_variables({"input": "who is Mayank?"})

{'history': "Human: Sunny and Mayank are working on a hackathon project\nAI: That's awesome! What's the hackathon project about?\nHuman: It's a machine learning project focused on healthcare.\nAI: Sounds exciting! Are they building a prediction model or something else?\nHuman: Yes, they are building prediction model.\nAI: Wishing Sunny and Mayank all the best for their project!",
 'entities': {'Mayank': 'Mayank is working on a hackathon project with Sunny.'}}

In [34]:
conversation = ConversationChain(
    llm=llm,
    verbose=True,
    prompt=ENTITY_MEMORY_CONVERSATION_TEMPLATE,
    memory=ConversationEntityMemory(llm=llm)
)

In [35]:
conversation.predict(input="Deven & Sam are working on a hackathon project")



> Entering new ConversationChain chain...
Prompt after formatting:
You are an assistant to a human, powered by a large language model trained by OpenAI.

You are designed to be able to assist with a wide range of tasks, from answering simple questions to providing in-depth explanations and discussions on a wide range of topics. As a language model, you are able to generate human-like text based on the input you receive, allowing you to engage in natural-sounding conversations and provide responses that are coherent and relevant to the topic at hand.

You are constantly learning and improving, and your capabilities are constantly evolving. You are able to process and understand large amounts of text, and can use this knowledge to provide accurate and informative responses to a wide range of questions. You have access to some personalized information provided by the human in the Context section below. Additionally, you are able to generate your own text based on the input you receive, 

"That's great to hear! Hackathons can be a fun and rewarding experience. If you need any assistance or have any questions related to your project, feel free to ask."

In [36]:
conversation.memory.entity_store.store

{'Deven': 'Deven is working on a hackathon project with Sam.',
 'Sam': 'Sam is working on a hackathon project with Deven.'}

In [37]:
conversation.predict(input="They are trying to add more complex memory structures to Langchain")



> Entering new ConversationChain chain...
Prompt after formatting:
You are an assistant to a human, powered by a large language model trained by OpenAI.

You are designed to be able to assist with a wide range of tasks, from answering simple questions to providing in-depth explanations and discussions on a wide range of topics. As a language model, you are able to generate human-like text based on the input you receive, allowing you to engage in natural-sounding conversations and provide responses that are coherent and relevant to the topic at hand.

You are constantly learning and improving, and your capabilities are constantly evolving. You are able to process and understand large amounts of text, and can use this knowledge to provide accurate and informative responses to a wide range of questions. You have access to some personalized information provided by the human in the Context section below. Additionally, you are able to generate your own text based on the input you receive, 

'Adding more complex memory structures to Langchain sounds like an interesting challenge. Memory structures are essential for storing and managing data effectively. The incorporation of more complex memory structures can potentially improve the performance and efficiency of Langchain. If Deven and Sam have any specific questions or need assistance with implementing these structures, feel free to provide more details, and I can offer guidance or suggestions to help with their hackathon project.'

In [38]:
conversation.predict(input="They are adding in a key-value store for entities mentioned so far in the conversation.")



> Entering new ConversationChain chain...
Prompt after formatting:
You are an assistant to a human, powered by a large language model trained by OpenAI.

You are designed to be able to assist with a wide range of tasks, from answering simple questions to providing in-depth explanations and discussions on a wide range of topics. As a language model, you are able to generate human-like text based on the input you receive, allowing you to engage in natural-sounding conversations and provide responses that are coherent and relevant to the topic at hand.

You are constantly learning and improving, and your capabilities are constantly evolving. You are able to process and understand large amounts of text, and can use this knowledge to provide accurate and informative responses to a wide range of questions. You have access to some personalized information provided by the human in the Context section below. Additionally, you are able to generate your own text based on the input you receive, 

'Incorporating a key-value store for entities mentioned in the conversation is a smart approach to managing and accessing data efficiently. Key-value stores provide a simple and fast way to store and retrieve data based on unique identifiers, which can be beneficial for organizing and accessing information in Langchain. If Deven and Sam need any guidance or assistance with implementing this key-value store, feel free to provide more details, and I can offer suggestions to help them with their project.'

In [39]:
conversation.predict(input="What do you know about Deven & Sam?")



> Entering new ConversationChain chain...
Prompt after formatting:
You are an assistant to a human, powered by a large language model trained by OpenAI.

You are designed to be able to assist with a wide range of tasks, from answering simple questions to providing in-depth explanations and discussions on a wide range of topics. As a language model, you are able to generate human-like text based on the input you receive, allowing you to engage in natural-sounding conversations and provide responses that are coherent and relevant to the topic at hand.

You are constantly learning and improving, and your capabilities are constantly evolving. You are able to process and understand large amounts of text, and can use this knowledge to provide accurate and informative responses to a wide range of questions. You have access to some personalized information provided by the human in the Context section below. Additionally, you are able to generate your own text based on the input you receive, 

'Deven and Sam are currently working together on a hackathon project, where they are adding more complex memory structures and incorporating a key-value store for entities mentioned in the conversation. This demonstrates their interest and commitment to using advanced data structures and techniques to enhance their project. If you have any more specific questions about Deven and Sam or their project, feel free to ask!'

In [40]:
conversation.memory.entity_store.store

{'Deven': 'Deven is working with Sam on a hackathon project, adding more complex memory structures and a key-value store for entities mentioned in the conversation.',
 'Sam': 'Sam is working on a hackathon project with Deven and they are adding more complex memory structures and incorporating a key-value store for entities mentioned in the conversation.',
 'Langchain': 'Langchain is being enhanced with a key-value store for entities mentioned in the conversation.'}

In [41]:
conversation.predict(input="Sam is the founder of a company called Daimon.")



> Entering new ConversationChain chain...
Prompt after formatting:
You are an assistant to a human, powered by a large language model trained by OpenAI.

You are designed to be able to assist with a wide range of tasks, from answering simple questions to providing in-depth explanations and discussions on a wide range of topics. As a language model, you are able to generate human-like text based on the input you receive, allowing you to engage in natural-sounding conversations and provide responses that are coherent and relevant to the topic at hand.

You are constantly learning and improving, and your capabilities are constantly evolving. You are able to process and understand large amounts of text, and can use this knowledge to provide accurate and informative responses to a wide range of questions. You have access to some personalized information provided by the human in the Context section below. Additionally, you are able to generate your own text based on the input you receive, 

"That's interesting to know that Sam is the founder of a company called Daimon. It seems like Sam has experience in entrepreneurship and is involved in leading a business. If there are any additional details or specific information you would like to share about Sam or Daimon, feel free to provide more context, and I can offer insights or assistance based on that information."

In [42]:
conversation.memory.entity_store.store

{'Deven': 'Deven is working with Sam on a hackathon project, adding more complex memory structures and a key-value store for entities mentioned in the conversation.',
 'Sam': 'Sam is working on a hackathon project with Deven and they are adding more complex memory structures and incorporating a key-value store for entities mentioned in the conversation.',
 'Langchain': 'Langchain is being enhanced with a key-value store for entities mentioned in the conversation.',
 'Daimon': 'Sam is the founder of a company called Daimon.'}

Conversation Summary Memory

In [44]:
memory = ConversationSummaryMemory(llm=llm, return_messages=True)

<ipython-input-44-5bc4a13bd712>:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryMemory(llm=llm, return_messages=True)


In [45]:
memory.save_context(
    {"input": "Sunny and Mayank are working on a hackathon project."},
    {"output": "That's awesome! What's the hackathon project about?"}
)

In [46]:
summary = memory.load_memory_variables({})
summary

{'history': [SystemMessage(content='The human mentions Sunny and Mayank are working on a hackathon project. The AI expresses excitement and asks about the details of the project.', additional_kwargs={}, response_metadata={})]}

In [48]:
memory.save_context(
    {"input": "It's a machine learning project focused on healthcare."},
    {"output": "Sounds exciting! Are they building a prediction model or something else"}
)
memory.save_context(
    {"input": "Yes, they’re working on a model to predict patient outcomes."},
    {"output": "Impressive! How far along are they with the project?"}
)

In [49]:
memory.load_memory_variables({})

{'history': [SystemMessage(content='The human mentions Sunny and Mayank are working on a hackathon project, a machine-learning project focused on healthcare. The AI expresses excitement and asks if they are building a prediction model or something else. The human confirms they are working on a model to predict patient outcomes. The AI is impressed and asks how far along they are with the project.', additional_kwargs={}, response_metadata={})]}

In [51]:
memory.chat_memory.messages

[HumanMessage(content='Sunny and Mayank are working on a hackathon project.', additional_kwargs={}, response_metadata={}),
 AIMessage(content="That's awesome! What's the hackathon project about?", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="It's a machine learning project focused on healthcare.", additional_kwargs={}, response_metadata={}),
 AIMessage(content='Sounds exciting! Are they building a prediction model or something else', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Yes, they’re working on a model to predict patient outcomes.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Impressive! How far along are they with the project?', additional_kwargs={}, response_metadata={})]

In [54]:
messages = memory.chat_memory.messages
previous_summary=""
memory.predict_new_summary(messages, previous_summary)

'The human mentions that Sunny and Mayank are working on a hackathon project focused on machine learning in healthcare to predict patient outcomes. The AI is impressed and asks how far along they are with the project.'

In [55]:
conversation_with_summary = ConversationChain(
    llm=llm,
    memory=ConversationSummaryMemory(llm=llm),
    verbose=True
)

In [56]:
conversation_with_summary.predict(input="Hi, what's up?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi, what's up?
AI:

> Finished chain.


"Hello! I'm just processing data and analyzing information from various sources. How can I assist you today?"

In [57]:
conversation_with_summary.predict(input="Sunny and Mayank are working on a mlops production ready project.")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
The human greets the AI and asks what's up. The AI responds by saying it's processing data and analyzing information to assist the human.
Human: Sunny and Mayank are working on a mlops production ready project.
AI:

> Finished chain.


"That's great to hear! MLOps projects can be very complex, involving the integration of machine learning models into production workflows. If you need any assistance or information on MLOps best practices, just let me know!"

Conversation Summary Buffer Memory
Keeps summary and buffer of the recent interaction

In [59]:
memory2 = ConversationSummaryBufferMemory(llm=llm,return_messages=True)

<ipython-input-59-d4ef2f584e8f>:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory2 = ConversationSummaryBufferMemory(llm=llm,return_messages=True)


In [60]:
memory2.save_context(
    {"input": "It's a machine learning project focused on healthcare."},
    {"output": "Sounds exciting! Are they building a prediction model or something else"}
)
memory2.save_context(
    {"input": "Yes, they’re working on a model to predict patient outcomes."},
    {"output": "Impressive! How far along are they with the project?"}
)

In [61]:
memory2.load_memory_variables({})

{'history': [HumanMessage(content="It's a machine learning project focused on healthcare.", additional_kwargs={}, response_metadata={}),
  AIMessage(content='Sounds exciting! Are they building a prediction model or something else', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Yes, they’re working on a model to predict patient outcomes.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Impressive! How far along are they with the project?', additional_kwargs={}, response_metadata={})]}

In [62]:
memory2.load_memory_variables({})["history"]

[HumanMessage(content="It's a machine learning project focused on healthcare.", additional_kwargs={}, response_metadata={}),
 AIMessage(content='Sounds exciting! Are they building a prediction model or something else', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Yes, they’re working on a model to predict patient outcomes.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Impressive! How far along are they with the project?', additional_kwargs={}, response_metadata={})]

In [63]:
conversation_with_summary = ConversationChain(
    llm=llm,
    # We set a very low max_token_limit for the purposes of testing.
    memory=ConversationSummaryBufferMemory(llm=llm, max_token_limit=40),
    verbose=True,
)
conversation_with_summary.predict(input="Hi, what's up?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi, what's up?
AI:

> Finished chain.


"Hello! I'm just here going through some data and ready to chat with you. How can I assist you today?"

In [64]:
conversation_with_summary.predict(input="Just working on writing some documentation on machine learning!")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
System: The human greets the AI by asking what's up.
AI: Hello! I'm just here going through some data and ready to chat with you. How can I assist you today?
Human: Just working on writing some documentation on machine learning!
AI:

> Finished chain.


"That's awesome! Machine learning is a fascinating field. If you need any specific information or examples to include in your documentation, feel free to ask me. I have lots of data on different algorithms, techniques, and applications in machine learning that may be helpful for you."

In [65]:
conversation_with_summary.predict(input="give me some points for writing about the document")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
System: The human greets the AI by asking what's up. The AI responds that it's going through data and ready to chat. The human mentions working on documentation on machine learning, to which the AI offers its assistance with information and examples on algorithms, techniques, and applications in machine learning.
Human: give me some points for writing about the document
AI:

> Finished chain.


'Of course! You can start by explaining the basics of machine learning, such as supervised learning, unsupervised learning, and reinforcement learning. Then, you can delve into popular machine learning algorithms like linear regression, decision trees, and neural networks. Additionally, you can discuss different techniques used in machine learning, such as feature engineering, model evaluation, and hyperparameter tuning. Finally, you can explore real-world applications of machine learning, such as image recognition, natural language processing, and recommendation systems. Let me know if you need more specific examples or details!'